# LC 424 — Longest Repeating Character Replacement

**Difficulty:** Medium  
**Category:** String  
**Pattern:** Sliding Window — Max Frequency Count

<div style="border-left: 4px solid purple; padding: 10px;
background-color: #f9f0ff; margin: 10px 0;">
<strong>Core Insight:</strong> A window is valid when
(window_size - max_freq) &lt;= k. Expand right freely;
shrink left only when that condition breaks.
</div>

## Official Problem Statement

You are given a string `s` and an integer `k`. You can choose
any character of the string and change it to any other
uppercase English character. You can perform this operation
at most `k` times.

Return the length of the longest substring containing the
same letter you can get after performing the above operations.

**Constraints:**
- `1 <= s.length <= 10^5`
- `s` consists of only uppercase English letters.
- `0 <= k <= s.length`

## What This Is Actually Asking

You have a string of uppercase letters. You can swap any
letter to anything else, up to k times total.
Goal: find the longest stretch where all letters are the same
after those swaps.
The trick is — you only care about the most frequent letter
in your current window. Everything else needs a swap.

## Walk Through an Example by Hand

Input: `s = "AABABBA"`, `k = 1`

```
Step 1: window='A'      freq={A:1}  max_f=1  size=1  1-1=0<=1 OK
Step 2: window='AA'     freq={A:2}  max_f=2  size=2  2-2=0<=1 OK
Step 3: window='AAB'    freq={A:2,B:1}  max_f=2  size=3
         3-2=1 <= 1  OK  (swap that 1 B)  best=3
Step 4: window='AABA'   freq={A:3,B:1}  max_f=3  size=4
         4-3=1 <= 1  OK  best=4
Step 5: window='AABAB'  freq={A:3,B:2}  max_f=3  size=5
         5-3=2 > 1   INVALID
         Shrink: remove s[left]='A'  left=1
         window='ABAB'  freq={A:2,B:2}  max_f=2  size=4
         4-2=2 > 1  still invalid
         Shrink: remove s[left]='A'  left=2
         window='BAB'   freq={A:1,B:2}  max_f=2  size=3
         3-2=1 <= 1  OK  best stays 4
Step 6: window='BABB'   freq={A:1,B:3}  max_f=3  size=4
         4-3=1 <= 1  OK  best=4
Step 7: window='BABBA'  freq={A:2,B:3}  max_f=3  size=5
         5-3=2 > 1  INVALID
         Shrink left to 3  window='ABBA'  size=4  OK
Answer: 4
```

## The Picture

Think of the window as a room. The dominant letter owns the
room. Everyone else is a stranger that costs one swap to
kick out. You have k eviction tickets.

```
s =  A  A  B  A  B  B  A
     0  1  2  3  4  5  6

  [L           R]    window='AABA'  size=4
   A  A  B  A        most frequent = A (count=3)
                     strangers = 4 - 3 = 1  <= k=1  VALID

  Formula:
  +---------------------------+
  | window_size - max_freq    |
  |    = swaps needed         |
  | if swaps_needed <= k:     |
  |   window is valid         |
  +---------------------------+

  max_freq never decreases — we only grow or hold
  the window, never shrink it smaller than best.
```

## When To Use This Pattern

- When you can **replace up to k elements** to make a window
  all the same.
- When the validity of a window depends on **how many
  non-dominant elements** it contains.
- When you want to maximize a window under a **budget
  constraint** (k replacements).
- When the problem says "at most k changes" + longest run.

## The Approach

Slide a window across the string. Count how often each
letter appears inside the window. At every step, check if
the number of non-dominant letters exceeds k — that is the
swap budget. If it does, move the left pointer one step
right. Track the largest valid window size seen.

In [ ]:
from typing import Dict  # for type hints in the solution

In [ ]:
def test_harness(func):
    """Run all test cases against func and print results."""
    tests = [
        # (s, k, expected)
        ("AABABBA",  1, 4),   # standard example
        ("ABAB",     2, 4),   # replace both B or both A
        ("AABA",     0, 2),   # no replacements allowed
        ("A",        0, 1),   # single char
        ("A",        1, 1),   # single char with budget
        ("AAAA",     2, 4),   # all same — whole string
        ("ABCDE",    1, 2),   # all different, k=1
        ("EAABBBCCE", 2, 6),  # multiple groups
        ("",          0, 0),  # empty string guard
    ]

    passed = 0
    for s, k, expected in tests:
        result = func(s, k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] s={repr(s):<12} k={k} "
            f"expected={expected}  got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def character_replacement(s: str, k: int) -> int:
    """
    Find the length of the longest substring where all
    characters are the same after at most k replacements.

    Approach:
    Use a sliding window. Track a frequency count of each
    letter inside the window. At each step, compute how many
    non-dominant letters are in the window — that is the
    replacements needed. If it exceeds k, move left forward
    by one. Always track the largest window seen.

    Time:  O(n) — right pointer visits each char once;
           left moves at most n steps total.
    Space: O(26) — frequency map holds at most 26 entries
           (uppercase English letters only).
    """
    pass


# Debug prints — run this cell while building your solution
print(character_replacement("AABABBA", 1))   # expected: 4
print(character_replacement("ABAB", 2))      # expected: 4
print(character_replacement("AABA", 0))      # expected: 2
print(character_replacement("", 0))          # expected: 0
print(character_replacement("AAAA", 2))      # expected: 4

In [ ]:
# Uncomment and run when solution is ready
# test_harness(character_replacement)

## Complexity

| Approach                  | Time   | Space  |
|---------------------------|--------|--------|
| Brute Force (all windows) | O(n^2) | O(26)  |
| Sliding Window            | O(n)   | O(26)  |

Brute force checks every possible start/end pair. Sliding
window avoids that by moving left forward just one step
at a time — never re-scanning from scratch.

## Real World Connection

At Citi, CloudWatch metrics from 6,000 endpoints stream
into Kinesis every 30 seconds. An AWS Lambda scans for
the longest window where at most k endpoints can be
reporting a non-critical status — meaning the cluster is
still healthy enough to keep serving traffic. DynamoDB
stores the rolling window results so on-call engineers
can see when tolerance thresholds were last breached.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra